# 01b — Batch Ingestion
**Ingest all past year papers from subject folders — with image extraction**

```
data/papers/
├── instrumentation_and_control/
│   ├── sept_2025.pdf
│   ├── jan_2025.pdf
│   └── sept_2024.pdf
└── another_subject/
    └── sept_2025.pdf
```
Per paper: Gemini extraction → parent-child chunks → Ollama embed
→ BM25 sparse → Pinecone upsert → PostgreSQL parents → Supabase images

---
## Section 1 — Setup

In [ ]:
import os, json, pickle, base64, time, hashlib, re
from pathlib import Path
from collections import defaultdict, deque
from dotenv import load_dotenv
load_dotenv()

import fitz                                      
import google.generativeai as genai
import ollama
import psycopg2
from psycopg2.extras import Json
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from supabase import create_client
from dataclasses import dataclass, field
from typing import Optional

# ── Clients ──────────────────────────────────────────────────────────────────
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model    = genai.GenerativeModel('gemma-4-31b-it')
pc       = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index    = pc.Index('papersloth')
supabase = create_client(os.getenv('SUPABASE_URL'), os.getenv('SUPABASE_KEY'))

# ── Constants ─────────────────────────────────────────────────────────────────
EMBED_MODEL          = 'nomic-embed-text-v2-moe:latest'
BUCKET               = 'exam-images'
PAPERS_ROOT          = Path('data/papers')
CHECKPOINT           = Path('data/checkpoint.json')
BM25_PATH            = Path('data/bm25_model.pkl')
ALL_EMBED_TEXTS_PATH = Path('data/all_embed_texts.json')
Path('data').mkdir(exist_ok=True)

stats = index.describe_index_stats()
print(f'✅ Setup complete')
print(f'   Pinecone : {stats["total_vector_count"]} vectors')
print(f'   Supabase : bucket = {BUCKET}')

---
## Section 2 — Folder Scanner

In [ ]:
def scan_papers(root: Path) -> list[dict]:
    papers = []
    for pdf in sorted(root.rglob('*.pdf')):
        subject = pdf.parent.name if pdf.parent != root else 'unknown'
        papers.append({
            'path':    str(pdf),
            'subject': subject,
            'name':    pdf.name,
            'key':     hashlib.md5(str(pdf).encode()).hexdigest()[:12],
        })
    return papers

if CHECKPOINT.exists():
    with open(CHECKPOINT) as f:
        done_keys = set(json.load(f).get('done', []))
else:
    done_keys = set()

all_papers = scan_papers(PAPERS_ROOT)
pending    = [p for p in all_papers if p['key'] not in done_keys]
already    = [p for p in all_papers if p['key'] in done_keys]

print(f'Found {len(all_papers)} PDFs  |  Already done: {len(already)}  |  Pending: {len(pending)}')
print()

by_subject = defaultdict(list)
for p in pending:
    by_subject[p['subject']].append(p['name'])
for subject, files in sorted(by_subject.items()):
    print(f'  📁 {subject}/')
    for f in files:
        print(f'       {f}')

eta = len(pending) * 5 / 60
print(f'\nEstimated time: ~{eta:.0f} min')

---
## Section 3 — Rate Limiter + Extraction Prompt

In [ ]:
class RateLimiter:
    """Sliding-window rate limiter. Blocks until a call slot is free."""
    def __init__(self, max_calls: int = 14, window_secs: float = 60.0):
        self.max_calls   = max_calls   # 14 not 15 — leave one buffer slot
        self.window_secs = window_secs
        self._calls      = deque()

    def wait(self):
        now = time.time()
        while self._calls and now - self._calls[0] > self.window_secs:
            self._calls.popleft()
        if len(self._calls) >= self.max_calls:
            sleep_for = self.window_secs - (now - self._calls[0]) + 0.5
            if sleep_for > 0:
                print(f'    ⏳ rate limit — sleeping {sleep_for:.1f}s')
                time.sleep(sleep_for)
        self._calls.append(time.time())

    @property
    def usage(self):
        now = time.time()
        return sum(1 for t in self._calls if now - t <= self.window_secs)

limiter = RateLimiter(max_calls=14, window_secs=60)
print('✅ Rate limiter ready')

In [ ]:
# Combined prompt — metadata + questions in ONE API call per paper
COMBINED_PROMPT = """This is a UTP (Universiti Teknologi PETRONAS) final exam paper.

Return a single JSON object with two keys: \"metadata\" and \"questions\".

\"metadata\": {
  \"course_code\":     \"RBB3013\",
  \"course_name\":     \"Instrumentation and Control\",
  \"semester\":        \"September 2025\",
  \"year\":            2025,
  \"duration_hours\":  3.0,
  \"total_questions\": 4
}

\"questions\": [
  {
    \"question_number\": \"1\",
    \"sub_question\":    \"a\",
    \"page_number\":     2,
    \"question_text\":   \"Full text...\",
    \"marks\":           10,
    \"has_figure\":      true,
    \"figure_label\":    \"FIGURE Q1b\",
    \"question_type\":   \"calculation\"
  }
]

Rules:
- Skip cover page, instructions, appendix
- Split sub-questions (a/b/c or i/ii/iii) into SEPARATE items
- Include full question text with all values and conditions
- Data tables → markdown format inside question_text
- question_type: calculation | theory | diagram | table
- has_figure = true if question references any FIGURE or TABLE
- page_number is 1-indexed
- marks / sub_question are null if not applicable

Return ONLY the JSON. No markdown fences. No extra text."""

print('✅ Extraction prompt ready')

---
## Section 4 — Processing Functions

All helpers used in the batch loop.

In [ ]:
# ── Data classes ──────────────────────────────────────────────────────────────
@dataclass
class ChildChunk:
    child_id:        str
    parent_id:       str
    question_number: str
    sub_question:    Optional[str]
    question_text:   str
    embed_text:      str
    marks:           Optional[int]
    has_figure:      bool
    figure_label:    Optional[str]
    question_type:   str
    course_code:     str
    semester:        str
    year:            int

@dataclass
class ParentChunk:
    parent_id:       str
    question_number: str
    full_text:       str
    total_marks:     Optional[int]
    children:        list
    course_code:     str
    semester:        str
    year:            int

print('✅ Data classes defined')

In [ ]:
def extract_pdf(pdf_path: str) -> dict:
    """Call Gemini once — returns metadata + questions."""
    with open(pdf_path, 'rb') as f:
        pdf_b64 = base64.standard_b64encode(f.read()).decode()
    limiter.wait()
    resp = model.generate_content([
        {'mime_type': 'application/pdf', 'data': pdf_b64},
        COMBINED_PROMPT
    ])
    raw = resp.text.strip()
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1].rsplit('```', 1)[0].strip()
    return json.loads(raw)


def build_chunks(extracted: dict) -> tuple:
    """Build parent + child objects from Gemini output."""
    meta        = extracted['metadata']
    questions   = extracted['questions']
    course_code = meta['course_code']
    semester    = meta['semester']
    year        = meta['year']
    sem_slug    = semester.replace(' ', '')

    groups  = defaultdict(list)
    for q in questions:
        groups[q['question_number']].append(q)

    parents:  list[ParentChunk] = []
    children: list[ChildChunk]  = []

    for qnum, subs in groups.items():
        parent_id  = f'{course_code}_{year}_{sem_slug}__Q{qnum}'
        full_parts = []
        child_ids  = []
        total_m    = 0

        for sub in subs:
            sub_l    = sub.get('sub_question') or ''
            child_id = f'{parent_id}_{sub_l}' if sub_l else f'{parent_id}_main'
            full_parts.append(
                f"Q{qnum}{sub_l}: {sub['question_text']}" +
                (f" [{sub['marks']} marks]" if sub.get('marks') else '')
            )
            child_ids.append(child_id)
            total_m += sub.get('marks') or 0

            preamble   = (
                f"Course: {course_code} | Semester: {semester} | "
                f"Question {qnum}{sub_l} | "
                f"Type: {sub.get('question_type','unknown')} | "
                f"Marks: {sub.get('marks','unknown')}\n"
            )
            children.append(ChildChunk(
                child_id        = child_id,
                parent_id       = parent_id,
                question_number = qnum,
                sub_question    = sub.get('sub_question'),
                question_text   = sub['question_text'],
                embed_text      = preamble + sub['question_text'],
                marks           = sub.get('marks'),
                has_figure      = sub.get('has_figure', False),
                figure_label    = sub.get('figure_label'),
                question_type   = sub.get('question_type', 'theory'),
                course_code     = course_code,
                semester        = semester,
                year            = year,
            ))

        parents.append(ParentChunk(
            parent_id       = parent_id,
            question_number = qnum,
            full_text       = '\n\n'.join(full_parts),
            total_marks     = total_m or None,
            children        = child_ids,
            course_code     = course_code,
            semester        = semester,
            year            = year,
        ))

    return parents, children


def extract_and_upload_images(
    pdf_path: str,
    questions: list[dict],
    course_code: str,
    semester: str,
    year: int,
) -> dict:
    """
    For every question with has_figure=True:
      - Render that PDF page as a 2x-resolution PNG (PyMuPDF)
      - Upload to Supabase Storage
      - Return {figure_label: public_url}
    """
    sem_slug    = semester.replace(' ', '')
    pdf_doc     = fitz.open(pdf_path)
    figure_urls = {}

    for q in questions:
        if not q.get('has_figure'):
            continue
        page_no = q.get('page_number')
        label   = q.get('figure_label') or f"Q{q['question_number']}{q.get('sub_question','')}"

        if not page_no:
            continue

        safe_label   = re.sub(r'[^\w\-]', '_', label)
        storage_path = f'{course_code}/{year}/{sem_slug}/{safe_label}.png'

        # Render page as PNG (2x zoom = ~150 DPI, clear for diagrams)
        page      = pdf_doc[page_no - 1]
        pix       = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
        img_bytes = pix.tobytes('png')

        try:
            supabase.storage.from_(BUCKET).upload(
                path=storage_path,
                file=img_bytes,
                file_options={'content-type': 'image/png', 'upsert': 'true'},
            )
        except Exception:
            pass  # upsert=true handles duplicates; other errors caught by caller

        url = supabase.storage.from_(BUCKET).get_public_url(storage_path)
        figure_urls[label] = url

    pdf_doc.close()
    return figure_urls  # { 'FIGURE Q1b': 'https://...', ... }


def upsert_to_pinecone(
    children:    list[ChildChunk],
    sparse_vecs: list,
) -> None:
    """Embed each child (Ollama) and upsert dense+sparse to Pinecone."""
    vectors = []
    for child, sparse in zip(children, sparse_vecs):
        resp  = ollama.embed(model=EMBED_MODEL, input=f'search_document: {child.embed_text}')
        dense = resp['embeddings'][0]
        vectors.append({
            'id':            child.child_id,
            'values':        dense,
            'sparse_values': sparse,
            'metadata': {
                'parent_id':       child.parent_id,
                'question_number': child.question_number,
                'sub_question':    child.sub_question or '',
                'marks':           child.marks or 0,
                'has_figure':      child.has_figure,
                'question_type':   child.question_type,
                'course_code':     child.course_code,
                'semester':        child.semester,
                'year':            child.year,
                'text_preview':    child.question_text[:200],
            },
        })
    for i in range(0, len(vectors), 100):
        index.upsert(vectors=vectors[i:i+100])


def upsert_parents_to_postgres(
    parents:     list[ParentChunk],
    figure_urls: dict,
) -> None:
    """
    Store each parent in PostgreSQL.
    image_urls are matched to each parent by question number:
      'FIGURE Q1b' → parent with question_number='1'
    """
    conn = psycopg2.connect(os.getenv('DATABASE_URL'))
    cur  = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS parent_chunks (
            parent_id TEXT PRIMARY KEY, question_number TEXT,
            full_text TEXT, total_marks INT, children JSONB,
            image_urls JSONB, course_code TEXT, semester TEXT,
            year INT, created_at TIMESTAMP DEFAULT NOW()
        );
    """)
    cur.execute("ALTER TABLE parent_chunks ADD COLUMN IF NOT EXISTS image_urls JSONB;")

    for p in parents:
        # Match figure labels that contain this question's number
        # e.g. 'FIGURE Q1b' matches question_number='1'
        q_imgs = {
            label: url
            for label, url in figure_urls.items()
            if re.search(rf'Q{p.question_number}[a-z]?', label, re.IGNORECASE)
        }
        cur.execute("""
            INSERT INTO parent_chunks
                (parent_id,question_number,full_text,total_marks,
                 children,image_urls,course_code,semester,year)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (parent_id) DO UPDATE SET
                full_text=EXCLUDED.full_text,
                total_marks=EXCLUDED.total_marks,
                children=EXCLUDED.children,
                image_urls=EXCLUDED.image_urls;
        """, (
            p.parent_id, p.question_number, p.full_text,
            p.total_marks, Json(p.children), Json(q_imgs),
            p.course_code, p.semester, p.year,
        ))

    conn.commit(); cur.close(); conn.close()


def save_checkpoint(done_keys: set) -> None:
    with open(CHECKPOINT, 'w') as f:
        json.dump({'done': list(done_keys)}, f)


print('✅ All functions defined')

---
## Section 5 — Batch Ingestion Loop

Per paper:
1. Gemini extraction (1 API call — rate limited)
2. Build parent/child chunks
3. Upload figure images → Supabase
4. Embed children → Pinecone
5. Store parents → PostgreSQL
6. Save checkpoint

In [ ]:
# Load existing embed texts for BM25 continuity
if ALL_EMBED_TEXTS_PATH.exists():
    with open(ALL_EMBED_TEXTS_PATH) as f:
        all_embed_texts: list[str] = json.load(f)
else:
    all_embed_texts = []

print(f'Starting. Existing corpus: {len(all_embed_texts)} texts.')

failed:  list[dict] = []
success: list[str]  = []

for paper in pending:
    path = paper['path']
    key  = paper['key']

    print(f"\n{'─'*60}")
    print(f"📄 {paper['subject']}/{paper['name']}  [RPM: {limiter.usage}/14]")

    try:
        t0 = time.time()

        # 1. Gemini — 1 call per paper ────────────────────────────────────
        extracted   = extract_pdf(path)
        meta        = extracted['metadata']
        course_code = meta['course_code']
        semester    = meta['semester']
        year        = meta['year']
        print(f'   ✅ Gemini  {course_code} {semester} '
              f'({len(extracted["questions"])} sub-q)  [{time.time()-t0:.1f}s]')

        # 2. Build chunks ──────────────────────────────────────────────────
        parents, children = build_chunks(extracted)
        new_texts = [c.embed_text for c in children]
        print(f'   ✅ Chunks  {len(parents)} parents  {len(children)} children')

        # 3. Images → Supabase ────────────────────────────────────────────
        figure_urls = extract_and_upload_images(
            path, extracted['questions'], course_code, semester, year
        )
        n_imgs = len(figure_urls)
        print(f'   ✅ Images  {n_imgs} uploaded to Supabase')

        # 4. BM25 sparse encoding ─────────────────────────────────────────
        all_embed_texts.extend(new_texts)
        bm25_local  = BM25Encoder()
        bm25_local.fit(all_embed_texts)
        sparse_vecs = bm25_local.encode_documents(new_texts)

        # 5. Embed + upsert to Pinecone ───────────────────────────────────
        upsert_to_pinecone(children, sparse_vecs)
        print(f'   ✅ Pinecone  {len(children)} vectors upserted')

        # 6. Parents → PostgreSQL ─────────────────────────────────────────
        upsert_parents_to_postgres(parents, figure_urls)
        print(f'   ✅ Postgres  {len(parents)} parents stored')

        # 7. Save checkpoint ──────────────────────────────────────────────
        done_keys.add(key)
        success.append(path)
        save_checkpoint(done_keys)
        with open(ALL_EMBED_TEXTS_PATH, 'w') as f:
            json.dump(all_embed_texts, f)

        print(f'   ⏱  {time.time()-t0:.1f}s total')

    except Exception as e:
        print(f'   ❌ FAILED: {e}')
        failed.append({'paper': paper, 'error': str(e)})


print(f"\n{'='*60}")
print(f'Done.  Success: {len(success)}  Failed: {len(failed)}')
if failed:
    for f in failed:
        print(f"  ❌ {f['paper']['name']}: {f['error']}")

In [ ]:
# ── Retry failed papers ───────────────────────────────────────────────────────
if not failed:
    print('No failures to retry.')
else:
    print(f'Retrying {len(failed)} papers...')
    retry_failed = []
    for entry in failed:
        paper = entry['paper']
        print(f"  ↩️  {paper['name']}")
        try:
            extracted         = extract_pdf(paper['path'])
            meta              = extracted['metadata']
            parents, children = build_chunks(extracted)
            new_texts         = [c.embed_text for c in children]
            figure_urls       = extract_and_upload_images(
                paper['path'], extracted['questions'],
                meta['course_code'], meta['semester'], meta['year']
            )
            all_embed_texts.extend(new_texts)
            bm25_r = BM25Encoder(); bm25_r.fit(all_embed_texts)
            upsert_to_pinecone(children, bm25_r.encode_documents(new_texts))
            upsert_parents_to_postgres(parents, figure_urls)
            done_keys.add(paper['key'])
            save_checkpoint(done_keys)
            print('     ✅ retry ok')
        except Exception as e:
            print(f'     ❌ still failing: {e}')
            retry_failed.append(entry)
    failed = retry_failed
    print(f'Remaining failures: {len(failed)}')

---
## Section 6 — Refit BM25 on Full Corpus

Run once after all papers are ingested.  
The saved model is what `02_retrieval.ipynb` loads at query time.

In [ ]:
print(f'Fitting BM25 on {len(all_embed_texts)} texts...')
bm25_final = BM25Encoder()
bm25_final.fit(all_embed_texts)
with open(BM25_PATH, 'wb') as f:
    pickle.dump(bm25_final, f)
print(f'✅ BM25 model saved → {BM25_PATH}')
print('   Reload in 02_retrieval.ipynb (re-run the BM25 load cell).')

---
## Section 7 — Report

In [ ]:
conn = psycopg2.connect(os.getenv('DATABASE_URL'))
cur  = conn.cursor()
cur.execute("""
    SELECT course_code, semester, year,
           COUNT(*) as questions,
           COUNT(CASE WHEN image_urls != '{}' THEN 1 END) as with_images
    FROM   parent_chunks
    GROUP BY course_code, semester, year
    ORDER BY course_code, year DESC;
""")
rows = cur.fetchall(); cur.close(); conn.close()
stats = index.describe_index_stats()

print('=' * 62)
print('PAPERSLOTH — INGESTION REPORT')
print('=' * 62)
print(f'{"SUBJECT":<12} {"SEMESTER":<22} {"YEAR":<6} {"Qs":<6} {"w/img"}')
print('-' * 62)
for r in rows:
    print(f'{r[0]:<12} {r[1]:<22} {r[2]:<6} {r[3]:<6} {r[4]}')
print('-' * 62)
print(f'Total questions in PostgreSQL : {sum(r[3] for r in rows)}')
print(f'Total vectors in Pinecone     : {stats["total_vector_count"]}')
print(f'BM25 corpus size              : {len(all_embed_texts)} texts')